In [0]:
%sql
-- ── 1. KPI Registry ───────────────────────────────────────────

CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc4_kpi_registry_b (

    kpi_id                      STRING,

    -- Which Gold table this KPI lives in
    gold_fully_qualified_name   STRING,
    business_date_column        STRING,

    -- KPI identity
    kpi_name                    STRING,
    kpi_column                  STRING,
    kpi_description             STRING,

    -- Formula validation
    formula_expression          STRING,
    formula_columns             STRING,    -- JSON array of columns used

    -- Range validation
    min_value                   DOUBLE,
    max_value                   DOUBLE,

    -- Anomaly detection
    baseline_window_days        INT,
    warning_deviation_pct       DOUBLE,
    critical_deviation_pct      DOUBLE,

    -- Which checks are active for this KPI
    checks_enabled              STRING,    -- JSON array

    -- Audit
    is_active                   BOOLEAN,
    registered_timestamp        TIMESTAMP,
    registered_by               STRING,
    change_reason               STRING

)
USING DELTA
COMMENT 'Defines each Gold KPI with formula, range bounds, and anomaly thresholds for Agent 4 validation.'

In [0]:
%sql
-- ── 2. KPI Baseline Metrics ───────────────────────────────────

CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc4_kpi_baseline_metrics_b (

    baseline_id                 STRING,

    gold_fully_qualified_name   STRING,
    kpi_name                    STRING,
    kpi_column                  STRING,

    baseline_date               DATE,
    baseline_window_days        INT,

    mean_value                  DOUBLE,
    stddev_value                DOUBLE,
    min_value                   DOUBLE,
    max_value                   DOUBLE,
    p25_value                   DOUBLE,
    p50_value                   DOUBLE,
    p75_value                   DOUBLE,
    p95_value                   DOUBLE,
    sample_count                INT,

    is_sufficient_data          BOOLEAN,
    baseline_status             STRING,

    computed_timestamp          TIMESTAMP,
    computed_by                 STRING

)
USING DELTA
COMMENT 'Historical baseline statistics for Gold KPIs used by Agent 4 anomaly detection.';


In [0]:
%sql
-- ── 3. KPI Validation Run ─────────────────────────────────────

CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc4_kpi_validation_run_b (

    validation_run_id           STRING,

    gold_fully_qualified_name   STRING,
    business_date               DATE,

    triggered_by                STRING,
    agent2_reconciliation_run_id STRING,

    validation_start_timestamp  TIMESTAMP,
    validation_end_timestamp    TIMESTAMP,
    validation_duration_seconds DOUBLE,

    overall_status              STRING,
    total_kpis_checked          INT,
    total_kpis_passed           INT,
    total_kpis_warned           INT,
    total_kpis_critical         INT,

    ai_enrichment_status        STRING,
    ai_enrichment_timestamp     TIMESTAMP,

    alert_sent                  BOOLEAN,
    alert_sent_timestamp        TIMESTAMP,
    alert_channel               STRING,

    created_timestamp           TIMESTAMP,
    created_by                  STRING

)
USING DELTA
PARTITIONED BY (business_date)
COMMENT 'One record per Agent 4 validation execution per Gold table per business date.';

In [0]:
%sql
-- ── 4. KPI Validation Results ─────────────────────────────────

CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc4_kpi_validation_results_b (

    result_id                   STRING,
    validation_run_id           STRING,

    gold_fully_qualified_name   STRING,
    business_date               DATE,

    kpi_name                    STRING,
    kpi_column                  STRING,
    check_type                  STRING,

    check_status                STRING,
    observed_value              DOUBLE,
    expected_value              DOUBLE,
    baseline_value              DOUBLE,
    deviation_percentage        DOUBLE,
    threshold_warning           DOUBLE,
    threshold_critical          DOUBLE,

    check_detail                STRING,

    ai_summary                  STRING,
    ai_recommended_action       STRING,
    ai_failure_category         STRING,
    ai_confidence_score         DOUBLE,
    ai_model_version            STRING,
    ai_invocation_id            STRING,

    created_timestamp           TIMESTAMP,
    ai_enriched_timestamp       TIMESTAMP

)
USING DELTA
PARTITIONED BY (business_date)
COMMENT 'One record per KPI per check per Agent 4 validation run.';

In [0]:
%sql
-- ── Verify ────────────────────────────────────────────────────

SHOW TABLES IN learning_dev.s819534 LIKE 'uc4_*';


database,tableName,isTemporary
s819534,uc4_kpi_baseline_metrics_b,false
s819534,uc4_kpi_registry_b,false
s819534,uc4_kpi_validation_results_b,false
s819534,uc4_kpi_validation_run_b,false


In [0]:
%sql
-- ══════════════════════════════════════════════════════════════
-- REGISTER KPIs FOR BOTH GOLD TABLES
-- ══════════════════════════════════════════════════════════════

INSERT INTO learning_dev.s819534.uc4_kpi_registry_b VALUES

-- ── Gold Daily Store Metrics KPIs ─────────────────────────────

-- return_rate formula check
(
    'kpi-store-001',
    'learning_dev.s819534.uc2_gold_daily_store_metrics',
    'business_date',
    'return_rate',
    'return_rate',
    'Proportion of transactions that are returns. Should equal return_count divided by total_transactions.',
    'return_count / total_transactions',
    '["return_count", "total_transactions", "return_rate"]',
    0.0,
    0.30,
    30,
    20.0,
    50.0,
    '["FORMULA", "RANGE", "ANOMALY"]',
    true,
    current_timestamp(),
    'agent4_admin',
    'Initial registration for Agent 4 PoC'
),

-- avg_basket_value formula check
(
    'kpi-store-002',
    'learning_dev.s819534.uc2_gold_daily_store_metrics',
    'business_date',
    'avg_basket_value',
    'avg_basket_value',
    'Average net amount per transaction. Should equal total_net_amount divided by total_transactions.',
    'total_net_amount / total_transactions',
    '["total_net_amount", "total_transactions", "avg_basket_value"]',
    0.0,
    5000.0,
    30,
    30.0,
    60.0,
    '["FORMULA", "RANGE", "ANOMALY"]',
    true,
    current_timestamp(),
    'agent4_admin',
    'Initial registration for Agent 4 PoC'
),

-- total_net_amount cross-KPI check
(
    'kpi-store-003',
    'learning_dev.s819534.uc2_gold_daily_store_metrics',
    'business_date',
    'net_vs_gross_consistency',
    'total_net_amount',
    'Net amount must be less than or equal to gross amount. Discount and tax should produce net <= gross.',
    'total_net_amount <= total_gross_amount',
    '["total_net_amount", "total_gross_amount"]',
    NULL,
    NULL,
    30,
    0.0,
    0.0,
    '["CROSS_KPI"]',
    true,
    current_timestamp(),
    'agent4_admin',
    'Initial registration for Agent 4 PoC'
),

-- return_count cross-KPI check
(
    'kpi-store-004',
    'learning_dev.s819534.uc2_gold_daily_store_metrics',
    'business_date',
    'return_count_consistency',
    'return_count',
    'Return count cannot exceed total transaction count.',
    'return_count <= total_transactions',
    '["return_count", "total_transactions"]',
    NULL,
    NULL,
    30,
    0.0,
    0.0,
    '["CROSS_KPI"]',
    true,
    current_timestamp(),
    'agent4_admin',
    'Initial registration for Agent 4 PoC'
),

-- total_net_amount anomaly check
(
    'kpi-store-005',
    'learning_dev.s819534.uc2_gold_daily_store_metrics',
    'business_date',
    'total_net_amount',
    'total_net_amount',
    'Total net revenue for the day across all stores. Monitor for unusual spikes or drops.',
    NULL,
    '["total_net_amount"]',
    0.0,
    10000000.0,
    30,
    25.0,
    50.0,
    '["ANOMALY", "RANGE"]',
    true,
    current_timestamp(),
    'agent4_admin',
    'Initial registration for Agent 4 PoC'
),

-- ── Gold Daily Inventory Summary KPIs ─────────────────────────

-- net_quantity_change formula check
(
    'kpi-inv-001',
    'learning_dev.s819534.uc2_gold_daily_inventory_summary',
    'business_date',
    'net_quantity_change',
    'net_quantity_change',
    'Net quantity movement. Should equal total_quantity_in minus total_quantity_out.',
    'total_quantity_in - total_quantity_out',
    '["total_quantity_in", "total_quantity_out", "net_quantity_change"]',
    -1000000.0,
    1000000.0,
    30,
    30.0,
    60.0,
    '["FORMULA", "ANOMALY"]',
    true,
    current_timestamp(),
    'agent4_admin',
    'Initial registration for Agent 4 PoC'
),

-- total_quantity_in cross-KPI check
(
    'kpi-inv-002',
    'learning_dev.s819534.uc2_gold_daily_inventory_summary',
    'business_date',
    'quantity_in_non_negative',
    'total_quantity_in',
    'Total quantity in must always be non-negative.',
    'total_quantity_in >= 0',
    '["total_quantity_in"]',
    0.0,
    NULL,
    30,
    0.0,
    0.0,
    '["CROSS_KPI"]',
    true,
    current_timestamp(),
    'agent4_admin',
    'Initial registration for Agent 4 PoC'
),

-- total_cost_impact anomaly check
(
    'kpi-inv-003',
    'learning_dev.s819534.uc2_gold_daily_inventory_summary',
    'business_date',
    'total_cost_impact',
    'total_cost_impact',
    'Total cost impact of all inventory movements for the day.',
    NULL,
    '["total_cost_impact"]',
    0.0,
    100000000.0,
    30,
    30.0,
    60.0,
    '["ANOMALY", "RANGE"]',
    true,
    current_timestamp(),
    'agent4_admin',
    'Initial registration for Agent 4 PoC'
);


-- Verify
SELECT
    kpi_id,
    SUBSTRING_INDEX(gold_fully_qualified_name, '.', -1) AS gold_table,
    kpi_name,
    kpi_column,
    checks_enabled,
    ROUND(warning_deviation_pct, 0)                     AS warn_pct,
    ROUND(critical_deviation_pct, 0)                    AS crit_pct,
    is_active
FROM learning_dev.s819534.uc4_kpi_registry_b
ORDER BY gold_fully_qualified_name, kpi_id;

kpi_id,gold_table,kpi_name,kpi_column,checks_enabled,warn_pct,crit_pct,is_active
kpi-inv-001,uc2_gold_daily_inventory_summary,net_quantity_change,net_quantity_change,"[""FORMULA"", ""ANOMALY""]",30.0,60.0,true
kpi-inv-002,uc2_gold_daily_inventory_summary,quantity_in_non_negative,total_quantity_in,"[""CROSS_KPI""]",0.0,0.0,true
kpi-inv-003,uc2_gold_daily_inventory_summary,total_cost_impact,total_cost_impact,"[""ANOMALY"", ""RANGE""]",30.0,60.0,true
kpi-store-001,uc2_gold_daily_store_metrics,return_rate,return_rate,"[""FORMULA"", ""RANGE"", ""ANOMALY""]",20.0,50.0,true
kpi-store-002,uc2_gold_daily_store_metrics,avg_basket_value,avg_basket_value,"[""FORMULA"", ""RANGE"", ""ANOMALY""]",30.0,60.0,true
kpi-store-003,uc2_gold_daily_store_metrics,net_vs_gross_consistency,total_net_amount,"[""CROSS_KPI""]",0.0,0.0,true
kpi-store-004,uc2_gold_daily_store_metrics,return_count_consistency,return_count,"[""CROSS_KPI""]",0.0,0.0,true
kpi-store-005,uc2_gold_daily_store_metrics,total_net_amount,total_net_amount,"[""ANOMALY"", ""RANGE""]",25.0,50.0,true
